In [1]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [2]:
#%load_ext pyinstrument

In [3]:
USE_NEGATIVE_WGT = False
MAX_ITER = 1

In [4]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-09-08-04_36_16_PM'

In [5]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/data"
    experiments_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/experiments"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    experiments_path = "../../experiments"
    output_path = "../../"

In [6]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [7]:
experiment_config = {
     "experiment": {
        "model": "hill_climb",
        "exp_oof_path": "2026-09-07-07_15_19_PM_optuna_xgboost",
        "description": "xgb hill climb sample",
        "use_negative_wgt": USE_NEGATIVE_WGT,
        "max_iter": MAX_ITER
    }
}
experiment_config["experiment"]["id"] = f"{dt_str}_{experiment_config["experiment"]["model"]}"

In [8]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
y = raw_train_df[target_column]

In [9]:
exp_oof_path = Path(experiments_path) / experiment_config["experiment"]["exp_oof_path"]
tune_trials_df = pd.read_csv(exp_oof_path / "optuna_trials.csv")

trial_names = []
oof_dfs = pd.DataFrame()
ss_dfs = pd.DataFrame()
oof_id = None
submission_id = None

for subdir in exp_oof_path.iterdir():
    if not subdir.is_dir():
        continue

    trial_names.append(subdir.name)
    oof_file = subdir / "oof.csv"

    if oof_file.exists():
        file_df = pd.read_csv(oof_file)
        if oof_id is None:
            oof_id = file_df['id']
        oof_dfs[subdir.name] = file_df[target_column]

    submission_file = subdir / "submission.csv"
    
    if submission_file.exists():
        file_df = pd.read_csv(submission_file)
        if submission_id is None:
            submission_id = file_df['id']
        ss_dfs[subdir.name] = file_df[target_column]

y_valid = y.iloc[oof_id]
oof_dfs

,xgboost_trial_15,xgboost_trial_14,xgboost_trial_11,xgboost_trial_24,xgboost_trial_39,xgboost_trial_43,xgboost_trial_6,xgboost_trial_0,xgboost_trial_37,xgboost_trial_40,...,xgboost_trial_29,xgboost_trial_21,xgboost_trial_19,xgboost_trial_1,xgboost_trial_38,xgboost_trial_20,xgboost_trial_27,xgboost_trial_8,xgboost_trial_30,xgboost_trial_41
0,0.408845,0.325952,0.524296,0.445312,0.235163,0.214934,0.190994,0.305121,0.428880,0.344099,...,0.048095,0.098805,0.323904,0.215241,0.217275,0.228657,0.328328,0.234677,0.225335,0.259407
1,0.900430,0.926197,0.979946,0.994472,0.993529,0.989139,0.988359,0.949019,0.971831,0.949256,...,0.996300,0.993071,0.974560,0.996324,0.970213,0.971033,0.954325,0.983707,0.992922,0.996347
2,0.999774,0.999376,0.999978,0.999767,0.999954,0.999791,0.999916,0.999957,0.999642,0.999657,...,0.999827,0.999881,0.999930,0.999924,0.998494,0.999739,0.999788,0.999818,0.999978,0.999990
3,0.996018,0.987900,0.998345,0.996406,0.999703,0.997089,0.999216,0.998327,0.996458,0.992772,...,0.999747,0.998548,0.998782,0.999822,0.993820,0.995897,0.992060,0.998546,0.999475,0.999871
4,0.587523,0.475495,0.455615,0.467223,0.535303,0.509671,0.462955,0.397003,0.369798,0.462860,...,0.423978,0.368453,0.455468,0.392243,0.440411,0.458874,0.422793,0.458753,0.481071,0.431547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138269,0.999341,0.997152,0.999828,0.998466,0.998768,0.999233,0.997471,0.999833,0.998946,0.995999,...,0.999210,0.998779,0.999486,0.999661,0.992992,0.998902,0.996520,0.998104,0.999165,0.999575
138270,0.997904,0.993622,0.997059,0.996726,0.997690,0.996121,0.994361,0.996711,0.997718,0.997114,...,0.994086,0.997496,0.994646,0.999453,0.993776,0.996584,0.996594,0.997064,0.994811,0.998760
138271,0.038707,0.193224,0.096514,0.070709,0.064817,0.050793,0.068306,0.075180,0.072699,0.202285,...,0.089187,0.063405,0.026462,0.108483,0.093228,0.031870,0.197306,0.071335,0.049009,0.065216
138272,0.332171,0.275086,0.137183,0.182174,0.199536,0.223935,0.236980,0.289177,0.306813,0.331691,...,0.334667,0.315286,0.261376,0.349518,0.352882,0.194285,0.275916,0.233200,0.154419,0.313353


In [10]:
best_start_model_idx = tune_trials_df['value'].idxmax()
ensemble_score = tune_trials_df.iloc[best_start_model_idx]['value']

In [11]:
start = -0.50
if not USE_NEGATIVE_WGT: 
    start = 0.01

weights = np.arange(start, 0.51, 0.01)
weights.shape

(50,)

In [12]:
y_valid_np = np.asarray(y_valid)
pos_mask = y_valid_np == 1

n_pos = pos_mask.sum()
n_neg = len(y_valid_np) - n_pos

def fast_auc_batch(y_true, predictions):
    ranks = rankdata(predictions, axis=0, method="average")
    pos_rank_sum = ranks[y_true == 1].sum(axis=0)
    auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

    return auc

In [13]:
iter = 0

models = [f'xgboost_trial_{best_start_model_idx}']
wgts = []
scores = [ensemble_score]

ensemble_oof = oof_dfs[models[0]].to_numpy().reshape(-1, 1)
submission_oof = ss_dfs[models[0]].to_numpy().reshape(-1, 1)

start_time = time.time()

while iter < MAX_ITER:
    model_valid_scores = []
    best_weight_idxs = []

    for model in oof_dfs.columns:
        model_oof = oof_dfs[model].to_numpy().reshape(-1, 1)

        weight_oofs = (
            model_oof * weights
            + ensemble_oof * (1 - weights)
        )

        weight_scores = fast_auc_batch(
            y_valid_np,
            weight_oofs
        )

        max_idx = np.argmax(weight_scores)
        max_weight_score = weight_scores[max_idx]

        model_valid_scores.append(max_weight_score)
        best_weight_idxs.append(max_idx)

    model_valid_scores = np.array(model_valid_scores)

    max_model_idx = np.argmax(model_valid_scores)
    max_weight_idx = best_weight_idxs[max_model_idx]

    candidate_score = model_valid_scores[max_model_idx]

    model_name = oof_dfs.columns[max_model_idx]
    candidate_weight = weights[max_weight_idx]

    candidate_oof = oof_dfs[model_name].to_numpy().reshape(-1, 1)
    candidate_ss = ss_dfs[model_name].to_numpy().reshape(-1, 1)

    if candidate_score <= ensemble_score:
        break

    ensemble_oof = (
        candidate_oof * candidate_weight
        + ensemble_oof * (1 - candidate_weight)
    )

    submission_oof = (
        candidate_ss * candidate_weight
        + submission_oof * (1 - candidate_weight)
    )

    ensemble_score = candidate_score

    models.append(model_name)
    wgts.append(candidate_weight)
    scores.append(candidate_score)

    iter += 1

    wgt = np.array([1.0])

    for w in wgts:
        wgt = wgt * (1 - w)
        wgt = np.concatenate([wgt, np.array([w])])

    model_weight_df = pd.DataFrame({
        'model': models,
        'weight': wgt,
        'ensemble_score': scores
    })
    print(model_weight_df)

elapsed = time.time() - start_time

              model  weight  ensemble_score
0  xgboost_trial_25    0.66        0.967550
1  xgboost_trial_49    0.34        0.967735


In [14]:
oof_df = pd.DataFrame({'id': oof_id, target_column: ensemble_oof.reshape(-1)})
ss_df = pd.DataFrame({'id': submission_id, target_column: submission_oof.reshape(-1)})

In [15]:
wgt = np.array([1.0])

for w in wgts:
    wgt = wgt * (1 - w)
    wgt = np.concatenate([wgt, np.array([w])])

In [16]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "exp_oof_path": experiment_config["experiment"]["exp_oof_path"],
    "use_negative_wgt": USE_NEGATIVE_WGT,
    "max_iter": MAX_ITER,
    "primary_metric": {
        "name": "auc",
        "value": round(scores[-1], 5)
    },
    "models": models,
    "weights": wgt.tolist(),
    "oof_scores": scores,
    "validation": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [17]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

model_weight_df.to_csv(experiment_path / "weights.csv", index=False)

oof_df.to_csv(experiment_path / "oof.csv", index=False)
ss_df.to_csv(experiment_path / f"submission.csv", index=False)